# Korea RIM (Residual Income Model) Valuation v1
## Sales-driven DuPont ROE → Ohlson (1995) Residual Income Valuation

### 처리 흐름
```
①  korea_fs_data_from_DG  →  과거 IS / BS / CF 분기 데이터
②  korea_revenue_forecast_result → 8분기 매출 예측
③  DuPont 분해 (Sales-driven)
      NPM  = NetIncome / Sales      [OLS slope 또는 median ratio]
      AT   = TTM Sales / Total Assets
      FL   = Total Assets / Equity
      ROE  = NPM × AT × FL          [Phase 1: 2yr + moat plateau]
④  Re   =  Rf(BOK) + β_blume × ERP  (E(Rm)=Damodaran 7% 또는 KOSPI geo)
⑤  RI spread AR(1) decay
      Phase 2 (5~30yr) : RI(t) = ω × RI(t-1)   [EVA/moat based ω]
      Phase 3          : TV = RI_last × (1+g) / (Re - g)
⑥  Intrinsic Value  =  BV₀  +  Σ PV(RI_t)  +  PV(TV)
      → 주당가치 = IV / shares_treasury_adj
```

### 학술 근거
- **Ohlson (1995)** *JAR* — Residual Income / Linear Information Model
- **Feltham & Ohlson (1995)** *JAR* — ω (persistence) coefficient
- **Fama & French (1995)** *JF* — Mean reversion of profitability (AR(1))
- **Mauboussin & Johnson (1997)** *FAJ* — Competitive Advantage Period
- **Damodaran (2002)** — 3-Stage 구조 및 phase period 지침

### 호영님 5가지 결정사항 (FCFF / RelVal 동일)
1. DataGuide item_code — xlsx 헤더 기반 매핑 35개
2. 매출예측 — `korea_revenue_forecast_result` (indicator=model명, created_at=예측일)
3. KOSPI — FDR/pykrx/yfinance 3단 fallback (Cell 3에서 패치 적용)
4. ERP — Damodaran 한국 7% 기본 / KOSPI geo 10y floor 옵션
5. 결측 처리 — 핵심(Revenue/OI/Equity) 결측 시 제외, 보조는 median fallback, **전부 `DataQualityReport` 추적** → `korea_valuation_quality_log` 저장

### US v11 → Korea v1 어댑테이션
| 항목 | US v11 | Korea v1 |
|---|---|---|
| 재무 데이터 | FMP API | `korea_fs_data_from_DG` |
| 매출 예측 | `us_revenue_forecast_data` | `korea_revenue_forecast_result` |
| Re | FMP bottom-up CAPM + size/country premium | 자체 베타 10y + Damodaran 한국 ERP |
| Re range | [7.5%, 18.0%] | **[6.0%, 18.0%]** (한국 저금리 반영) |
| GDP growth | 4.0% | **2.5%** |
| TV_RE_GAP | 0.75% | **1.0%** (한국 기업 장기성장 보수적) |
| 음수 BV 대응 | FMP totalDebt 기반 IC | DataGuide DEBT_KEYS 합산 IC |
| Retention | 배당 + 자사주매입 | **배당만** (한국 buyback 데이터 제한적) |
| 현재가 / 주식수 | FMP quote / weightedAvgShs | `KSE_Price` / DG `S420004400` |

### 저장
- 결과: `korea_rim_valuation` (PK: ticker, date, year_label)
- 품질: `korea_valuation_quality_log` (model='RIM')


## Cell 1 · 경로 자동 감지

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",
]

def _setup_path() -> str:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root 자동 감지 : {root}")
            return root
    for cand in _CANDIDATE_ROOTS:
        if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "DATA")):
            if cand not in sys.path:
                sys.path.insert(0, cand)
            print(f"[PATH] root 후보 경로 : {cand}")
            return cand
    raise EnvironmentError("DATA 폴더를 찾을 수 없습니다.")

_ROOT = _setup_path()
print(f"[확인] 프로젝트 루트 : {_ROOT}")


[PATH] root 자동 감지 : C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast
[확인] 프로젝트 루트 : C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast


## Cell 2 · Import & 설정 상수

> **여기만 수정하면 됩니다**:
> - `TICKER_START / TICKER_END / SKIP_DONE`
> - `ERP_METHOD`
> - `MIN_REVENUE_QUARTERS`, `PHASE1_EXTRA_YR`


In [2]:
import gc, math, time, traceback
from datetime import datetime, date, timedelta
from typing import Optional, Dict, Any, List, Tuple

import numpy as np
import pandas as pd
import pymysql
from scipy import stats
from IPython.display import display
import matplotlib
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sqlalchemy import text

from DATA.config import get_db_info, get_engine
from DATA.KEYS import KEYS
from DATA.korea_valuation_helpers import (
    setup_project_path, to_dg_ticker, to_price_ticker, get_pymysql_conn,
    DG_ITEM_CODES,
    load_korea_financials_wide, load_korea_revenue_forecast,
    load_korea_marketcap_latest, load_korea_price_series, load_current_price,
    load_kospi_series, get_risk_free_rate, compute_beta_10y,
    estimate_market_return, get_universe_with_min_history,
    DataQualityReport, save_quality_report_to_db,
)

def log(tag, msg):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}][{tag}] {msg}", flush=True)

# ══════════════════════════════════════════════════════════════════
#  DB 테이블
# ══════════════════════════════════════════════════════════════════
TABLE_FS        = "korea_fs_data_from_DG"
TABLE_FORECAST  = "korea_revenue_forecast_result"
TABLE_PRICE     = "KSE_Price"
TABLE_MARKETCAP = "ks_listed_company_daily_marketcap"
TABLE_RESULT    = "korea_rim_valuation"
TABLE_QUALITY   = "korea_valuation_quality_log"

# ══════════════════════════════════════════════════════════════════
#  모델 파라미터
# ══════════════════════════════════════════════════════════════════
FORECAST_HORIZON    = 8       # 예측 분기 수 (= 2년)
MIN_HISTORY         = 12      # DuPont OLS 최소 분기 수
MIN_REVENUE_QUARTERS = 24     # universe 필터: 최소 6년 매출 보유

# OLS 기준 (ratio 회귀가 본질적으로 noisy → FCFF 보다 완화)
OLS_MIN_R2          = 0.20
OLS_MIN_SAMPLES     = 12
WINSORIZE_LIMITS    = (0.05, 0.95)

# ── Terminal / Phase 2 ─────────────────────────────────────────
GDP_GROWTH          = 0.025   # 한국 장기 GDP
TV_RE_GAP           = 0.010   # TV: g = Re - 1% (US v11은 0.75%)
RETENTION_FLOOR     = -0.50   # 음수 허용 (buyback > NI 예외 케이스)
BV_RETENTION_CAP    = 0.75

# ── Phase 1 기간: moat 등급별 plateau ──────────────────────────
PHASE1_BASE_YR  = 2   # Sales forecast 실제 예측 2년 (고정)
PHASE1_EXTRA_YR = {
    "Exceptional moat":  5,   # 총 7yr
    "Wide moat":         4,   # 총 6yr
    "Wide-Narrow moat":  3,   # 총 5yr
    "Narrow moat":       2,   # 총 4yr
    "Some moat":         1,   # 총 3yr
    "No moat":           0,   # 총 2yr
    "Unknown (fallback)":1,
}
PHASE1_YR_BY_MOAT = {g: PHASE1_BASE_YR + e for g, e in PHASE1_EXTRA_YR.items()}

# ── Phase 1 ROE 안정장치 ───────────────────────────────────────
PHASE1_ROE_FLOOR_RATIO   = 0.80    # 과거 TTM ROE 대비 floor 비율
PHASE1_ROE_CEILING_RATIO = 2.50

# ── WACC / Re ──────────────────────────────────────────────────
ERP_METHOD        = "damodaran_floor"   # 'damodaran_floor' | 'kospi_geo_10y'
DAMODARAN_ERP_KR  = 0.07
GEO_FLOOR         = 0.07
RF_FALLBACK       = 0.035
RE_FLOOR          = 0.06    # 한국 저금리 반영 (US v11은 7.5%)
RE_CAP            = 0.18
RD_DEFAULT        = 0.045

# ── 단위 변환 ──────────────────────────────────────────────────
FS_UNIT_MULTIPLIER        = 1_000        # 천원 → 원
MARKETCAP_UNIT_MULTIPLIER = 1_000_000    # 백만원 → 원

# ── 재무 항목 키 ───────────────────────────────────────────────
DEBT_KEYS = ["short_term_debt", "current_lt_debt", "bonds",
             "long_term_debt", "lease_liab"]
CASH_KEYS = ["cash", "short_term_invest"]

# ── 배치 범위 ──────────────────────────────────────────────────
TICKER_START = 0
TICKER_END   = 9999
SKIP_DONE    = False

CHECKPOINT_DIR = "_korea_rim_checkpoint"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
DONE_PATH = os.path.join(CHECKPOINT_DIR, "done_tickers.txt")
FAIL_PATH = os.path.join(CHECKPOINT_DIR, "failed_tickers.txt")

db_info = get_db_info()
engine  = get_engine(db_info)

print("[OK] Import 완료")
print(f"[설정] FORECAST_HORIZON={FORECAST_HORIZON}Q  ERP_METHOD={ERP_METHOD}")
print(f"[설정] GDP_GROWTH={GDP_GROWTH}  TV_RE_GAP={TV_RE_GAP}  Re∈[{RE_FLOOR:.0%}, {RE_CAP:.0%}]")
print(f"[설정] Phase1 plateau by moat (기본 2yr + moat 연장):")
for k, v in PHASE1_EXTRA_YR.items():
    print(f"        {k:<22s}: {PHASE1_BASE_YR}+{v} = {PHASE1_BASE_YR+v}yr")


[OK] Import 완료
[설정] FORECAST_HORIZON=8Q  ERP_METHOD=damodaran_floor
[설정] GDP_GROWTH=0.025  TV_RE_GAP=0.01  Re∈[6%, 18%]
[설정] Phase1 plateau by moat (기본 2yr + moat 연장):
        Exceptional moat      : 2+5 = 7yr
        Wide moat             : 2+4 = 6yr
        Wide-Narrow moat      : 2+3 = 5yr
        Narrow moat           : 2+2 = 4yr
        Some moat             : 2+1 = 3yr
        No moat               : 2+0 = 2yr
        Unknown (fallback)    : 2+1 = 3yr


## Cell 3 · DB 연결 & 결과 테이블 초기화

결과 테이블 `korea_rim_valuation`:
- PK = (ticker, date, year_label) — 같은 종목+평가일자에서 Ph1 year1/year2/... + Ph2 year1/year2/... + TV row 각각 저장
- 평가일자(date)가 달라지면 별도 row → 시계열 추적 가능


In [3]:
CREATE_SQL = f"""
CREATE TABLE IF NOT EXISTS `{TABLE_RESULT}` (
  `id`              BIGINT      NOT NULL AUTO_INCREMENT,
  `date`            DATE        NOT NULL COMMENT '평가 실행일',
  `ticker`          VARCHAR(20) NOT NULL,
  `year_label`      VARCHAR(10)          COMMENT '2026 / Ph2-Y3 / TV',
  `phase`           VARCHAR(10)          COMMENT 'ph1 / ph1e / ph2 / tv',
  `sales_forecast`  DOUBLE               COMMENT '연간 매출 (원)',
  `npm_forecast`    DOUBLE,
  `asset_turnover`  DOUBLE,
  `fin_leverage`    DOUBLE,
  `roe_forecast`    DOUBLE,
  `re`              DOUBLE               COMMENT 'Cost of Equity',
  `ri_spread`       DOUBLE               COMMENT 'ROE - Re',
  `bv_start`        DOUBLE               COMMENT 'BV 또는 IC (원)',
  `ri`              DOUBLE               COMMENT 'Residual Income (원)',
  `pv_ri`           DOUBLE               COMMENT 'PV of RI',
  `moat_label`      VARCHAR(30),
  `rho`             DOUBLE               COMMENT 'AR(1) omega',
  `n_phase2`        INT,
  `g_terminal`      DOUBLE,
  `pv_all_ri`       DOUBLE,
  `terminal_value`  DOUBLE,
  `intrinsic_value` DOUBLE,
  `current_price`   DOUBLE,
  `upside_pct`      DOUBLE,
  `target_price`    DOUBLE,
  `beta_raw`        DOUBLE,
  `beta_blume`      DOUBLE,
  `bv_source`       VARCHAR(10)          COMMENT 'Equity / IC_fallback',
  `revenue_quarters`INT,
  `forecast_model`  VARCHAR(20),
  `forecast_date`   DATE,
  `created_at`      DATETIME    DEFAULT CURRENT_TIMESTAMP,
  PRIMARY KEY (`id`),
  UNIQUE KEY uq_main (`ticker`, `date`, `year_label`),
  INDEX idx_ticker (`ticker`),
  INDEX idx_date   (`date`)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
"""

conn = get_pymysql_conn(db_info)
try:
    with conn.cursor() as cur:
        cur.execute(CREATE_SQL)
    conn.commit()
    log("DB", f"테이블 준비 완료: {TABLE_RESULT}")
finally:
    conn.close()

try:
    with engine.connect() as c:
        c.execute(text("SELECT 1"))
    log("DB", f"연결 성공 host={db_info.get('host')} port={db_info.get('port')}")
except Exception as e:
    log("DB", f"연결 실패: {e}")

# Universe 조회
universe_df = get_universe_with_min_history(
    db_info, min_quarters=MIN_REVENUE_QUARTERS, require_consecutive=False)
KOREA_TICKER_LIST = universe_df["ticker"].tolist()
print(f"\n[Universe] 매출 ≥{MIN_REVENUE_QUARTERS}분기 종목: {len(KOREA_TICKER_LIST):,}개")


[14:13:16][DB] 테이블 준비 완료: korea_rim_valuation
[14:13:16][DB] 연결 성공 host=hystox74.synology.me port=3307

[Universe] 매출 ≥24분기 종목: 1,267개


## Cell 4 · 시장 파라미터 — Rf, KOSPI, E(Rm)

FCFF/RelVal 노트북과 동일 로직. KOSPI 로더는 3단 fallback (FDR → pykrx → yfinance).


In [4]:
# 1. Risk-free (BOK 10Y 국고채)
RF, RF_SOURCE = get_risk_free_rate(KEYS["BOK"], fallback_rate=RF_FALLBACK)
log("MKT", f"Rf = {RF:.4%}  (source: {RF_SOURCE})")

# 2. KOSPI (helpers의 3단 fallback 로더 사용)
KOSPI_PX = load_kospi_series(
    start_date=(datetime.today() - timedelta(days=365*12)).strftime("%Y-%m-%d"))
log("MKT", f"KOSPI {len(KOSPI_PX):,}거래일  "
            f"({KOSPI_PX.index.min().date()} ~ {KOSPI_PX.index.max().date()})")

# 3. E(Rm)
mkt = estimate_market_return(
    method=ERP_METHOD, rf=RF, kospi_series=KOSPI_PX,
    years=10, damodaran_erp_kr=DAMODARAN_ERP_KR, geo_floor=GEO_FLOOR)
E_RM = mkt["e_rm"]
ERP  = mkt["erp"]
log("MKT", f"E(Rm) = {E_RM:.4%}  ERP = {ERP:.4%}  ({mkt['note']})")

print()
print("=" * 60)
print(f"  Rf     = {RF:>7.3%}")
print(f"  ERP    = {ERP:>7.3%}")
print(f"  E(Rm)  = {E_RM:>7.3%}")
print(f"  Re 범위 = [{RE_FLOOR:.0%}, {RE_CAP:.0%}]")
print("=" * 60)


[14:13:23][MKT] Rf = 3.8170%  (source: BOK_2026-04-24)
[KOSPI] try source=fdr (2014-04-29 ~ 2026-04-26) ... FAIL (FDR 실패 (재시도 3회): LOGOUT)
[KOSPI] try source=pykrx (2014-04-29 ~ 2026-04-26) ... FAIL ('지수명')
[KOSPI] try source=yfinance (2014-04-29 ~ 2026-04-26) ... OK  2,937거래일
[14:13:34][MKT] KOSPI 2,937거래일  (2014-04-29 ~ 2026-04-24)
[14:13:34][MKT] E(Rm) = 10.8170%  ERP = 7.0000%  (Damodaran 한국 ERP 7.0% 적용)

  Rf     =  3.817%
  ERP    =  7.000%
  E(Rm)  = 10.817%
  Re 범위 = [6%, 18%]


## Cell 5 · KoreaRIMModel 클래스

US v11 RIMModel 의 한국화. 핵심 로직 (Ohlson 1995, 6-Tier moat, AR(1) decay) 동일, 데이터 소스만 DataGuide DB로 교체.


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  KoreaRIMModel — Ohlson (1995) Residual Income Valuation
# ═══════════════════════════════════════════════════════════════

class KoreaRIMModel:
    """
    한국 주식 RIM 평가 모델.

    공식:
        IV = BV_0 + Σ RI_t / (1+Re)^t + PV(TV)
        RI_t = (ROE_t - Re) × BV_{t-1}
        BV_t = BV_{t-1} × (1 + ROE_t × retention)  [with Re floor]

    Phase 1: DuPont 기반 ROE forecast (2yr 실제 + moat plateau 확장)
    Phase 2: AR(1) decay  RI(t) = ω × RI(t-1)   [5~30yr moat 등급별]
    Phase 3: TV = RI_last × (1+g) / (Re - g)

    US v11과 동일한 6-Tier moat 분류 + OR 로직 + n_pos 페널티.
    """

    def __init__(self, ticker, engine, db_info, rf, e_rm, kospi_series,
                 forecast_horizon=FORECAST_HORIZON,
                 min_history=MIN_HISTORY,
                 gdp_growth=GDP_GROWTH,
                 verbose=False):
        self.ticker_dg    = to_dg_ticker(ticker)
        self.ticker_price = to_price_ticker(ticker)
        self.engine       = engine
        self.db_info      = db_info
        self.rf           = rf
        self.e_rm         = e_rm
        self.erp          = e_rm - rf
        self.kospi        = kospi_series
        self.horizon      = forecast_horizon
        self.min_history  = min_history
        self.gdp_growth   = gdp_growth
        self.verbose      = verbose

        self._sales_actual   = None
        self._sales_forecast = None
        self._used_model     = ""
        self._forecast_date  = None
        self._fs_wide        = None
        self._re             = None
        self._eva_cache      = None
        self._beta_info      = None

        self._using_ic    = False
        self._n_phase1    = 2
        self._bv_source   = "Equity"
        self.result_df    = None
        self.valuation    = None

        self.report = DataQualityReport(ticker=self.ticker_dg)

    # ─────────────────────────────────────────────────────────
    # Utility
    # ─────────────────────────────────────────────────────────
    @staticmethod
    def _winsorize(s, limits=WINSORIZE_LIMITS):
        s = s.dropna()
        if len(s) < 4: return s
        lo, hi = s.quantile(limits[0]), s.quantile(limits[1])
        return s.clip(lo, hi)

    @staticmethod
    def _ols_ratio(x, y):
        mask = x.notna() & y.notna() & (x != 0)
        if mask.sum() < OLS_MIN_SAMPLES:
            return np.nan, -1.0, int(mask.sum())
        slope, _, r, _, _ = stats.linregress(x[mask], y[mask])
        return float(slope), float(r**2), int(mask.sum())

    @staticmethod
    def _n(val):
        if val is None: return None
        try:
            f = float(val)
            return None if (np.isnan(f) or np.isinf(f)) else f
        except (TypeError, ValueError):
            return val

    # ─────────────────────────────────────────────────────────
    # 1. Data Loading
    # ─────────────────────────────────────────────────────────
    def load_sales(self):
        wide = load_korea_financials_wide(
            self.ticker_dg, self.db_info, table_name=TABLE_FS,
            item_keys=["revenue"], fillna_zero=False)
        actual = wide["revenue"].dropna() * FS_UNIT_MULTIPLIER
        if actual.empty or len(actual) < MIN_REVENUE_QUARTERS:
            self.report.add("revenue_actual", "missing", n_obs=len(actual),
                            note=f"actual {len(actual)}Q < {MIN_REVENUE_QUARTERS}")
            raise ValueError(
                f"[{self.ticker_dg}] 매출 actual 부족 ({len(actual)}Q < {MIN_REVENUE_QUARTERS}Q)")
        self.report.add("revenue_actual", "ok", n_obs=len(actual))

        forecast, model_name, ca = load_korea_revenue_forecast(
            self.ticker_dg, self.db_info, table_name=TABLE_FORECAST,
            horizon=self.horizon)
        if forecast.empty:
            self.report.add("revenue_forecast", "missing", n_obs=0,
                            note="korea_revenue_forecast_result 없음")
            raise ValueError(f"[{self.ticker_dg}] 매출 forecast 없음")
        forecast = forecast * FS_UNIT_MULTIPLIER
        self.report.add("revenue_forecast", "ok", n_obs=len(forecast),
                        note=f"model={model_name}")

        self._sales_actual   = actual
        self._sales_forecast = forecast.iloc[:self.horizon]
        self._used_model     = model_name
        self._forecast_date  = ca

        if self.verbose:
            log(self.ticker_dg,
                f"Sales actual={len(actual)}Q forecast={len(self._sales_forecast)}Q "
                f"({model_name})")
        return self

    def load_financials(self):
        wide = load_korea_financials_wide(
            self.ticker_dg, self.db_info, table_name=TABLE_FS,
            item_keys=None, fillna_zero=False)
        if wide.empty:
            raise ValueError(f"[{self.ticker_dg}] FS 데이터 없음")

        non_share = [c for c in wide.columns
                     if c not in ("shares_treasury_adj", "shares_common")]
        wide[non_share] = wide[non_share] * FS_UNIT_MULTIPLIER

        # 핵심항목 검증
        missing_core = []
        if "revenue" not in wide.columns or wide["revenue"].dropna().empty:
            missing_core.append("revenue")
        if "net_income" not in wide.columns or wide["net_income"].dropna().empty:
            missing_core.append("net_income")
        if "total_equity" not in wide.columns or wide["total_equity"].dropna().empty:
            missing_core.append("total_equity")
        if missing_core:
            for k in missing_core:
                self.report.add(k, "missing", n_obs=0)
            raise ValueError(f"[{self.ticker_dg}] 핵심 항목 결측: {missing_core}")

        self.report.add("fs_core", "ok",
                        n_obs=int(wide[["revenue","net_income","total_equity"]]
                                  .notna().all(axis=1).sum()))
        self._fs_wide = wide
        if self.verbose:
            log(self.ticker_dg, f"FS wide shape={wide.shape}")
        return self

    # ─────────────────────────────────────────────────────────
    # 2. Re (Cost of Equity) — 자체 베타 10y + CAPM
    # ─────────────────────────────────────────────────────────
    def load_re(self):
        """β_blume = 0.67×|β| + 0.33  →  Re = Rf + β_blume × ERP"""
        info = compute_beta_10y(
            self.ticker_dg, self.db_info, kospi_series=self.kospi,
            years=10, min_obs=750)
        self._beta_info = info

        if np.isnan(info["beta_raw"]):
            beta_blume = 1.0
            self.report.add("beta", "fallback_median", n_obs=info["n_obs"],
                            value=beta_blume, note="베타 계산 실패 → 1.0")
        else:
            beta_blume = info["beta_blume"]
            self.report.add("beta", "ok", n_obs=info["n_obs"],
                            value=beta_blume, r2=info["r_squared"],
                            note=f"β_raw={info['beta_raw']:.3f}")

        re = float(np.clip(self.rf + beta_blume * self.erp, RE_FLOOR, RE_CAP))
        self._re = re

        if self.verbose:
            log(self.ticker_dg,
                f"Re={re:.4%}  β_raw={info.get('beta_raw', np.nan):.3f} "
                f"β_blume={beta_blume:.3f}  Rf={self.rf:.3%} ERP={self.erp:.3%}")
        return self

    # ─────────────────────────────────────────────────────────
    # 3. 보조 함수: IC / TTM ROE / Retention / Tax
    # ─────────────────────────────────────────────────────────
    def _estimate_ic(self):
        """
        Invested Capital = Equity + Total Debt
        음수 BV 기업(적자 누적 또는 대량 자사주매입) 처리용.
        모든 부채가 없으면 totalAssets × 0.4 fallback.
        """
        wide = self._fs_wide.sort_index()
        last = wide.iloc[-1]

        eq = last.get("total_equity", 0) or 0
        td = 0.0
        for k in DEBT_KEYS:
            if k in wide.columns:
                v = last.get(k, 0)
                if pd.notna(v):
                    td += float(v)
        ic = float(eq) + td
        if ic <= 0:
            ta = last.get("total_assets", 0)
            ic = float(ta) * 0.40 if pd.notna(ta) and ta > 0 else 1e11
        return max(ic, 1e10)  # 최소 100억

    def _get_historical_roe_ttm(self):
        """최근 TTM ROE 계산 (DuPont blend용)"""
        try:
            wide = self._fs_wide.sort_index()
            if "net_income" not in wide.columns or "total_equity" not in wide.columns:
                return np.nan
            df = wide[["net_income", "total_equity"]].dropna()
            df = df[df["total_equity"] > 0]
            if len(df) < 4:
                return np.nan
            ni_ttm = float(df["net_income"].iloc[-4:].sum())
            eq_avg = float(df["total_equity"].iloc[-4:].mean())
            return ni_ttm / eq_avg if eq_avg > 0 else np.nan
        except Exception:
            return np.nan

    def _estimate_tax_rate(self):
        """실효세율 (한국 법정세율 22% fallback)"""
        wide = self._fs_wide
        if "pretax_income" not in wide.columns or "tax_expense" not in wide.columns:
            return 0.22
        df = wide[["pretax_income", "tax_expense"]].dropna()
        df = df[df["pretax_income"] > 0]
        if df.empty:
            return 0.22
        return float((df["tax_expense"] / df["pretax_income"]).clip(0, 0.40).median())

    def estimate_retention(self):
        """
        실질 이익유보율 = 1 - 배당성향.
        한국 기업은 buyback 데이터 신뢰성 낮음 → 배당(dividends_paid)만 사용.
        Clean Surplus 역산 fallback (NI - ΔEquity).
        """
        wide = self._fs_wide.sort_index()
        if "net_income" not in wide.columns:
            self.report.add("retention", "fallback_zero", value=0.70,
                            note="net_income 없음")
            return 0.70
        ni = wide["net_income"]
        if (ni > 0).sum() == 0:
            self.report.add("retention", "fallback_zero", value=0.70,
                            note="NI 양수 분기 0")
            return 0.70

        div = None
        # 방법 1: dividends_paid 직접 사용
        if "dividends_paid" in wide.columns:
            d = wide["dividends_paid"].abs()
            if (d > 0).sum() >= 4:
                div = d

        # 방법 2: Clean Surplus 역산 (NI - ΔEquity)
        if div is None and "total_equity" in wide.columns:
            eq = wide["total_equity"]
            delta_eq = eq.diff()
            implied = (ni - delta_eq).clip(lower=0)
            if (implied > 0).sum() >= 4:
                div = implied

        if div is None:
            self.report.add("retention", "fallback_zero", value=0.70,
                            note="배당/clean surplus 모두 실패")
            return 0.70

        payout = (div / ni.abs()).replace([np.inf, -np.inf], np.nan)
        po = payout.iloc[-8:].dropna()
        if po.empty:
            po = payout.dropna()
        if po.empty:
            self.report.add("retention", "fallback_zero", value=0.70)
            return 0.70

        payout_med = float(po.clip(0, 3.0).median())
        retention = float(np.clip(1.0 - payout_med, RETENTION_FLOOR, 0.98))

        self.report.add("retention", "ok", n_obs=len(po), value=retention,
                        note=f"payout_med={payout_med:.2%}")
        if self.verbose:
            log(self.ticker_dg, f"Retention={retention:.3f}  payout={payout_med:.3f}")
        return retention

    # ─────────────────────────────────────────────────────────
    # 4. DuPont 계수 추정
    # ─────────────────────────────────────────────────────────
    def estimate_dupont_coefs(self):
        """NPM (OLS → median), AT (TTM), FL (median)"""
        wide = self._fs_wide

        # ── NPM = NetIncome / Revenue (OLS slope) ────────────
        df_npm = wide[["revenue", "net_income"]].dropna()
        df_npm = df_npm[df_npm["revenue"] > 0]
        npm_series = self._winsorize((df_npm["net_income"] / df_npm["revenue"]).dropna())
        npm_median = float(npm_series.median()) if not npm_series.empty else 0.05
        npm_coef, npm_method = npm_median, "median"
        r2_npm = -1.0

        if len(df_npm) >= OLS_MIN_SAMPLES:
            slope, r2_npm, n = self._ols_ratio(df_npm["revenue"], df_npm["net_income"])
            if not np.isnan(slope) and r2_npm >= OLS_MIN_R2:
                npm_coef, npm_method = float(slope), "ols"
                self.report.add("npm", "ok", n_obs=n, value=npm_coef,
                                r2=r2_npm, note="OLS slope")
            else:
                self.report.add("npm", "fallback_median", n_obs=n,
                                value=npm_median, r2=r2_npm,
                                note=f"R²={r2_npm:.2f} → median")
        else:
            self.report.add("npm", "fallback_median", n_obs=len(df_npm),
                            value=npm_median,
                            note=f"n={len(df_npm)} < {OLS_MIN_SAMPLES}")

        # ── AT = TTM Sales / TotalAssets (median) ────────────
        at_median = 0.70
        if "total_assets" in wide.columns:
            df_at = wide[["revenue", "total_assets"]].dropna().sort_index()
            df_at = df_at[df_at["total_assets"] > 0]
            if len(df_at) >= 4:
                df_at["ttm"] = df_at["revenue"].rolling(4).sum()
                atm = df_at.dropna(subset=["ttm"])
                ratios = self._winsorize((atm["ttm"] / atm["total_assets"]).dropna())
                if not ratios.empty:
                    at_median = float(ratios.median())
                    self.report.add("asset_turnover", "ok",
                                    n_obs=len(ratios), value=at_median)
                else:
                    self.report.add("asset_turnover", "fallback_zero", value=at_median)
            else:
                self.report.add("asset_turnover", "fallback_zero", value=at_median,
                                note="total_assets < 4Q")
        else:
            self.report.add("asset_turnover", "fallback_zero", value=at_median,
                            note="total_assets 컬럼 없음")

        # ── FL = TotalAssets / Equity (median, clip [1, 20]) ──
        fl_median = 2.5
        if "total_assets" in wide.columns and "total_equity" in wide.columns:
            df_fl = wide[["total_assets", "total_equity"]].dropna()
            df_fl = df_fl[(df_fl["total_assets"] > 0) & (df_fl["total_equity"] > 0)]
            if not df_fl.empty:
                ratios = self._winsorize(
                    (df_fl["total_assets"] / df_fl["total_equity"]).dropna())
                fl_median = float(np.clip(ratios.median(), 1.0, 20.0))
                self.report.add("financial_leverage", "ok",
                                n_obs=len(ratios), value=fl_median)
            else:
                self.report.add("financial_leverage", "fallback_zero", value=fl_median,
                                note="equity>0 분기 없음")
        else:
            self.report.add("financial_leverage", "fallback_zero", value=fl_median)

        if self.verbose:
            log(self.ticker_dg,
                f"DuPont  NPM={npm_coef:.4f}({npm_method})  "
                f"AT={at_median:.3f}(TTM)  FL={fl_median:.2f}")

        return {
            "npm_coef":   npm_coef,
            "npm_method": npm_method,
            "npm_r2":     r2_npm,
            "at_median":  at_median,
            "fl_median":  fl_median,
        }

    # ─────────────────────────────────────────────────────────
    # 5. Phase 1 ROE forecast
    # ─────────────────────────────────────────────────────────
    def forecast_roe_phase1(self, coefs, n_years=2):
        """
        Phase 1 = 실제 매출 예측 구간 (forecast) + moat plateau 연장 구간 (gdp_ext).
        연장 구간에서는 Forecast 마지막 ROE를 고정 유지 (Mauboussin CAP 반영).
        """
        fc_q = self._sales_forecast
        re = self._re

        # Sales 연간 집계 (실제 예측)
        annual_sales_raw = []
        for yr in range(0, len(fc_q), 4):
            chunk = fc_q.iloc[yr: yr + 4]
            annual_sales_raw.append({
                "year": fc_q.index[yr].year,
                "sales": float(chunk.sum()),
                "source": "forecast",
            })

        last_yr = annual_sales_raw[-1]["year"] if annual_sales_raw else datetime.now().year
        last_sal = annual_sales_raw[-1]["sales"] if annual_sales_raw else 0.0

        # Plateau 연장 (GDP 복리)
        annual_sales = list(annual_sales_raw)
        for i in range(len(annual_sales_raw), n_years):
            extra = i - len(annual_sales_raw) + 1
            annual_sales.append({
                "year": last_yr + extra,
                "sales": last_sal * (1 + GDP_GROWTH) ** extra,
                "source": "gdp_ext",
            })

        # BV₀ 확보 — 음수면 IC로 대체
        wide = self._fs_wide.sort_index()
        eq_ser = wide["total_equity"].dropna()
        bv0_raw = float(eq_ser.iloc[-1]) if not eq_ser.empty else -1.0
        if bv0_raw <= 0:
            bv0 = self._estimate_ic()
            self._using_ic = True
            self._bv_source = "IC_fallback"
            self.report.add("bv0", "fallback_median", n_obs=0, value=bv0,
                            note=f"음수 BV({bv0_raw/1e12:.2f}조) → IC 대체")
            if self.verbose:
                log(self.ticker_dg,
                    f"BV 음수 → IC={bv0/1e12:.2f}조 사용")
        else:
            bv0 = bv0_raw
            self._using_ic = False
            self._bv_source = "Equity"
            self.report.add("bv0", "ok", value=bv0)

        retention = self.estimate_retention()
        hist_roe = self._get_historical_roe_ttm()

        rows = []
        bv_start = bv0
        roe_fixed = None

        for yr_info in annual_sales:
            is_flat = (yr_info.get("source", "forecast") == "gdp_ext")

            if not is_flat:
                sales = yr_info["sales"]
                ni = coefs["npm_coef"] * sales
                npm = ni / sales if sales > 0 else 0.0
                at_est = coefs["at_median"]
                fl_est = coefs["fl_median"]
                roe_raw = npm * at_est * fl_est

                if not np.isnan(hist_roe) and hist_roe > 0:
                    # blend 65/35 + floor/ceiling
                    roe_blend = 0.65 * roe_raw + 0.35 * hist_roe
                    floor_roe = hist_roe * PHASE1_ROE_FLOOR_RATIO
                    ceil_roe  = hist_roe * PHASE1_ROE_CEILING_RATIO
                    if roe_raw < floor_roe:
                        roe = max(roe_blend, floor_roe)
                    elif roe_raw > ceil_roe:
                        roe = min(roe_blend, hist_roe * 1.50)
                    else:
                        roe = roe_blend
                    roe = float(np.clip(roe, -0.99, 3.0))
                else:
                    roe = float(np.clip(roe_raw, -0.99, 2.5))

                roe_fixed = roe
            else:
                roe = roe_fixed if roe_fixed is not None else (hist_roe or re + 0.05)
                sales = yr_info["sales"]
                ni = coefs["npm_coef"] * sales
                npm = ni / sales if sales > 0 else 0.0
                at_est = coefs["at_median"]
                fl_est = coefs["fl_median"]

            ri_spread = roe - re
            ri = ri_spread * bv_start

            rows.append({
                "year":         yr_info["year"],
                "phase":        "ph1" if not is_flat else "ph1e",
                "sales_annual": sales,
                "net_income":   ni,
                "npm":          npm,
                "at":           at_est,
                "fl":           fl_est,
                "roe":          roe,
                "re":           re,
                "ri_spread":    ri_spread,
                "bv_start":     bv_start,
                "ri":           ri,
            })
            # BV 진행: BV × (1 + Re × b_bv) + RI × b_bv
            b_bv = retention if retention < 0 else min(retention, BV_RETENTION_CAP)
            bv_start = bv_start * (1.0 + re * b_bv) + ri * b_bv

        return pd.DataFrame(rows), bv_start

    # ─────────────────────────────────────────────────────────
    # 6. EVA spread + Moat 분류 (6-Tier)
    # ─────────────────────────────────────────────────────────
    def compute_wacc_simple(self):
        """EVA 계산용 간단 WACC"""
        re = self._re or 0.10
        tax = self._estimate_tax_rate()
        wide = self._fs_wide

        # Rd
        rd = RD_DEFAULT
        if "interest_expense" in wide.columns:
            total_debt = pd.Series(0.0, index=wide.index)
            for k in DEBT_KEYS:
                if k in wide.columns:
                    total_debt = total_debt + wide[k].fillna(0)
            td_avg = (total_debt + total_debt.shift(1)) / 2
            ie = wide["interest_expense"].abs()
            valid = (td_avg > 0) & ie.notna()
            if valid.sum() >= 2:
                rd = float((ie[valid] / td_avg[valid]).clip(0, 0.20).median())

        # 시가총액
        mc, _ = load_korea_marketcap_latest(
            self.ticker_dg, self.db_info, table_name=TABLE_MARKETCAP)
        if mc is None or mc <= 0:
            return re  # fallback: Re만 사용
        mkt_cap = mc * MARKETCAP_UNIT_MULTIPLIER

        # 총부채 (latest)
        total_debt_latest = 0.0
        for k in DEBT_KEYS:
            if k in wide.columns:
                v = wide[k].dropna()
                if not v.empty:
                    total_debt_latest += float(v.iloc[-1])

        V = mkt_cap + total_debt_latest
        if V <= 0:
            return re
        wacc = re * (mkt_cap/V) + rd * (1-tax) * (total_debt_latest/V)
        return float(np.clip(wacc, 0.04, 0.25))

    def compute_eva_spread(self):
        """EVA spread = ROIC_TTM - WACC, plus ROE-Re spread"""
        wacc = self.compute_wacc_simple()
        tax = self._estimate_tax_rate()
        wide = self._fs_wide

        if "operating_income" not in wide.columns:
            return {"roic": np.nan, "wacc": wacc, "eva_spread": np.nan,
                    "roe_re_spread": np.nan, "n_positive": 0, "eva_series": []}

        # NOPAT quarterly
        nopat_q = wide["operating_income"] * (1 - tax)

        # IC = Equity + Total Debt - Cash
        td_s = pd.Series(0.0, index=wide.index)
        for k in DEBT_KEYS:
            if k in wide.columns:
                td_s = td_s + wide[k].fillna(0)
        cash_s = pd.Series(0.0, index=wide.index)
        for k in CASH_KEYS:
            if k in wide.columns:
                cash_s = cash_s + wide[k].fillna(0)
        ic_s = wide["total_equity"].fillna(0) + td_s - cash_s

        merged = pd.concat([nopat_q.rename("nopat_q"), ic_s.rename("ic")],
                           axis=1, join="inner").dropna().sort_index()
        if len(merged) < 4:
            return {"roic": np.nan, "wacc": wacc, "eva_spread": np.nan,
                    "roe_re_spread": np.nan, "n_positive": 0, "eva_series": []}

        eva_series = []
        dates = sorted(merged.index)
        for i, dt in enumerate(dates):
            if i < 3: continue
            nopat_ttm = float(merged["nopat_q"].iloc[i-3:i+1].sum())
            ic_snap = float(merged["ic"].iloc[i])
            if ic_snap <= 0: continue
            roic_q = nopat_ttm / ic_snap
            eva_q = roic_q - wacc
            eva_series.append({"date": dt, "roic": roic_q, "eva_spread": eva_q})

        if not eva_series:
            return {"roic": np.nan, "wacc": wacc, "eva_spread": np.nan,
                    "roe_re_spread": np.nan, "n_positive": 0, "eva_series": []}

        latest = eva_series[-1]
        recent_20 = eva_series[-20:]
        n_pos = sum(1 for e in recent_20 if e["eva_spread"] > 0)

        # ROE - Re spread
        re = self._re or 0.10
        roe_re_spread = np.nan
        df_re = wide[["net_income", "total_equity"]].dropna()
        df_re = df_re[df_re["total_equity"] > 0]
        if len(df_re) >= 4:
            ni_ttm = float(df_re["net_income"].iloc[-4:].sum())
            eq_avg = float(df_re["total_equity"].iloc[-4:].mean())
            if eq_avg > 0:
                roe_ttm = ni_ttm / eq_avg
                roe_re_spread = float(roe_ttm - re)

        result = {
            "roic":          latest["roic"],
            "wacc":          wacc,
            "eva_spread":    latest["eva_spread"],
            "roe_re_spread": roe_re_spread,
            "n_positive":    n_pos,
            "eva_series":    eva_series,
        }
        self._eva_cache = result

        if self.verbose:
            rrs_str = f"{roe_re_spread*100:+.1f}%" if not np.isnan(roe_re_spread) else "N/A"
            log(self.ticker_dg,
                f"EVA: ROIC={latest['roic']:.2%} WACC={wacc:.2%} "
                f"spread={latest['eva_spread']:+.2%} n_pos={n_pos}/20  ROE-Re={rrs_str}")
        return result

    def moat_to_omega_and_years(self, eva):
        """
        6-Tier 분류 (US v11 동일):
          Exceptional / Wide / Wide-Narrow / Narrow / Some / No moat
          각 등급별 (omega, n_phase2) + n_pos 페널티 + Phase1 plateau.
        """
        eva_spread    = eva.get("eva_spread",    np.nan)
        roe_re_spread = eva.get("roe_re_spread", np.nan)
        n_pos         = eva.get("n_positive", 0)

        if np.isnan(eva_spread) and np.isnan(roe_re_spread):
            return 0.760, 8, "Unknown (fallback)", PHASE1_YR_BY_MOAT["Unknown (fallback)"]

        eva_s = eva_spread    if not np.isnan(eva_spread)    else -np.inf
        roe_s = roe_re_spread if not np.isnan(roe_re_spread) else -np.inf

        # 1차 OR 분류
        GRADE_PARAMS = [
            ("Exceptional moat",  0.990, 30, 0.25, 1.50),
            ("Wide moat",         0.980, 28, 0.15, 0.80),
            ("Wide-Narrow moat",  0.970, 25, 0.10, 0.40),
            ("Narrow moat",       0.950, 20, 0.06, 0.20),
            ("Some moat",         0.940, 15, 0.03, 0.05),
        ]
        grade, omega, n = "No moat", 0.880, 10
        for label, w, yr, eva_th, roe_th in GRADE_PARAMS:
            if eva_s > eva_th or roe_s > roe_th:
                grade, omega, n = label, w, yr
                break

        # n_pos 페널티
        N_POS_MIN = {
            "Exceptional moat":  18, "Wide moat": 16, "Wide-Narrow moat": 13,
            "Narrow moat": 10, "Some moat": 7, "No moat": 0,
        }
        ORDER = ["Exceptional moat", "Wide moat", "Wide-Narrow moat",
                 "Narrow moat", "Some moat", "No moat"]
        ALL_PARAMS = {
            "Exceptional moat":  (0.990, 30), "Wide moat": (0.980, 28),
            "Wide-Narrow moat":  (0.970, 25), "Narrow moat": (0.950, 20),
            "Some moat":         (0.940, 15), "No moat":    (0.880, 10),
        }
        if n_pos < N_POS_MIN.get(grade, 0):
            idx = ORDER.index(grade)
            if idx < len(ORDER) - 1:
                grade = ORDER[idx + 1]
                omega, n = ALL_PARAMS[grade]

        n_phase1 = PHASE1_YR_BY_MOAT.get(grade, 2)
        return omega, n, grade, n_phase1

    # ─────────────────────────────────────────────────────────
    # 7. RI Path + Valuation
    # ─────────────────────────────────────────────────────────
    def compute_ri_path(self):
        re = self._re
        coefs = self.estimate_dupont_coefs()
        eva = self.compute_eva_spread()
        omega, n_phase2, moat_label, n_phase1 = self.moat_to_omega_and_years(eva)

        ph1_df, bv_after_ph1 = self.forecast_roe_phase1(coefs, n_years=n_phase1)
        retention = self.estimate_retention()

        rows = ph1_df.to_dict("records")

        # Phase 2: AR(1) decay
        ri_prev = float(ph1_df["ri"].iloc[-1]) if not ph1_df.empty else 0.0
        bv_current = bv_after_ph1
        last_year = int(ph1_df["year"].iloc[-1]) if not ph1_df.empty else datetime.now().year

        for t in range(1, n_phase2 + 1):
            ri_t = omega * ri_prev
            ri_spread = ri_t / bv_current if bv_current > 0 else 0.0
            roe_impl = ri_spread + re

            rows.append({
                "year":         last_year + t,
                "phase":        "ph2",
                "sales_annual": np.nan, "net_income": np.nan,
                "npm": np.nan, "at": np.nan, "fl": np.nan,
                "roe": roe_impl, "re": re,
                "ri_spread": ri_spread,
                "bv_start":  bv_current,
                "ri":        ri_t,
            })
            ri_prev = ri_t
            b_bv = retention if retention < 0 else min(retention, BV_RETENTION_CAP)
            bv_current = bv_current * (1.0 + re * b_bv) + ri_t * b_bv

        self.result_df = pd.DataFrame(rows)
        self._bv_terminal = bv_current
        self._ri_last = ri_prev
        self._omega = omega
        self._moat_label = moat_label
        self._n_phase2 = n_phase2
        self._n_phase1 = len(ph1_df)
        self._eva = eva

        if self.verbose:
            pv_factor = sum(omega**t / (1+re)**t for t in range(1, n_phase2+1))
            log(self.ticker_dg,
                f"RI path: Ph1={len(ph1_df)}yr Ph2={n_phase2}yr  "
                f"RI_last={ri_prev/1e12:.2f}조  "
                f"[{moat_label} ω={omega} PVf={pv_factor:.2f}]")
        return self

    def compute_valuation(self):
        re = self._re
        g = self.gdp_growth
        df = self.result_df.copy()

        # BV₀
        wide = self._fs_wide.sort_index()
        eq_ser = wide["total_equity"].dropna()
        bv0_raw = float(eq_ser.iloc[-1]) if not eq_ser.empty else -1.0
        bv0 = bv0_raw if bv0_raw > 0 else self._estimate_ic()

        # PV(RI)
        pv_ri_total = 0.0
        pv_rows = []
        for t, (_, row) in enumerate(df.iterrows(), 1):
            pv = row["ri"] / (1 + re) ** t
            pv_ri_total += pv
            pv_rows.append(pv)
        df["pv_ri"] = pv_rows

        T = len(df)

        # Terminal Value (US v11 방식: RI_last × (1+g) / (Re-g))
        ri_last = self._ri_last
        g_tv = max(re - TV_RE_GAP, self.gdp_growth)
        g_tv = min(g_tv, re - 0.005)
        tv = 0.0
        if ri_last > 0 and re > g_tv:
            tv = ri_last * (1.0 + g_tv) / (re - g_tv)
        elif ri_last > 0:
            tv = ri_last * (1.0 + g_tv) / 0.005
        pv_tv = tv / (1 + re) ** T if tv != 0 else 0.0

        intrinsic = bv0 + pv_ri_total + pv_tv

        # Shares
        shares = np.nan
        for col in ["shares_treasury_adj", "shares_common"]:
            if col in wide.columns:
                s = wide[col].dropna()
                s = s[s > 0]
                if not s.empty:
                    shares = float(s.iloc[-1])
                    break
        if np.isnan(shares):
            self.report.add("shares", "missing", n_obs=0)
            target_price = np.nan
        else:
            target_price = intrinsic / shares

        # Current Price
        cp = load_current_price(self.ticker_dg, self.db_info, table_name=TABLE_PRICE)
        if cp is None:
            self.report.add("current_price", "missing", n_obs=0)
            cp = np.nan

        upside = ((target_price/cp) - 1) * 100 \
                 if (not np.isnan(target_price) and not np.isnan(cp) and cp > 0) else np.nan
        tv_wt = pv_tv / intrinsic * 100 if intrinsic != 0 else np.nan

        self.result_df = df
        self.valuation = {
            "ticker": self.ticker_dg,
            "re": re, "g_terminal": g_tv,
            "bv0": bv0, "bv_source": self._bv_source,
            "pv_ri": pv_ri_total, "terminal_value": tv, "pv_tv": pv_tv,
            "tv_weight_pct": tv_wt,
            "intrinsic_value": intrinsic,
            "shares": shares, "target_price": target_price,
            "current_price": cp, "upside_pct": upside,
            "moat_label": self._moat_label, "omega": self._omega,
            "n_phase1": self._n_phase1, "n_phase2": self._n_phase2,
            "eva_spread": self._eva.get("eva_spread", np.nan),
            "roic": self._eva.get("roic", np.nan),
            "roe_re_spread": self._eva.get("roe_re_spread", np.nan),
            "beta_raw": self._beta_info.get("beta_raw") if self._beta_info else np.nan,
            "beta_blume": self._beta_info.get("beta_blume") if self._beta_info else np.nan,
            "revenue_quarters": len(self._sales_actual),
            "forecast_model": self._used_model,
            "forecast_date": self._forecast_date,
        }

        if self.verbose:
            tp_s = f"{target_price:,.0f}원" if not np.isnan(target_price) else "N/A"
            cp_s = f"{cp:,.0f}원" if not np.isnan(cp) else "N/A"
            up_s = f"{upside:+.1f}%" if not np.isnan(upside) else "N/A"
            log(self.ticker_dg,
                f"RIM  BV={bv0/1e12:.2f}조 PV(RI)={pv_ri_total/1e12:.2f}조 "
                f"PV(TV)={pv_tv/1e12:.2f}조 IV={intrinsic/1e12:.2f}조  "
                f"TP={tp_s} CP={cp_s} Up={up_s}  [{self._moat_label}]")
        return self

    # ─────────────────────────────────────────────────────────
    # 8. DB 저장
    # ─────────────────────────────────────────────────────────
    def save_to_db(self, run_date=None):
        if self.result_df is None or self.valuation is None:
            return 0
        run_date = run_date or datetime.now().strftime("%Y-%m-%d")
        v = self.valuation
        n = self._n
        df = self.result_df.copy()

        rows = []
        for _, row in df.iterrows():
            rows.append({
                "date": run_date, "ticker": self.ticker_dg,
                "year_label": str(row.get("year", "")),
                "phase": row.get("phase", ""),
                "sales_forecast": n(row.get("sales_annual")),
                "npm_forecast":   n(row.get("npm")),
                "asset_turnover": n(row.get("at")),
                "fin_leverage":   n(row.get("fl")),
                "roe_forecast":   n(row.get("roe")),
                "re":             n(row.get("re")),
                "ri_spread":      n(row.get("ri_spread")),
                "bv_start":       n(row.get("bv_start")),
                "ri":             n(row.get("ri")),
                "pv_ri":          n(row.get("pv_ri")),
                "moat_label":     v.get("moat_label"),
                "rho":            n(v.get("omega")),
                "n_phase2":       n(v.get("n_phase2")),
                "g_terminal":     n(v.get("g_terminal")),
                "pv_all_ri":      n(v.get("pv_ri")),
                "terminal_value": n(v.get("terminal_value")),
                "intrinsic_value":n(v.get("intrinsic_value")),
                "current_price":  n(v.get("current_price")),
                "upside_pct":     n(v.get("upside_pct")),
                "target_price":   n(v.get("target_price")),
                "beta_raw":       n(v.get("beta_raw")),
                "beta_blume":     n(v.get("beta_blume")),
                "bv_source":      v.get("bv_source"),
                "revenue_quarters": v.get("revenue_quarters"),
                "forecast_model": v.get("forecast_model"),
                "forecast_date":  v.get("forecast_date"),
            })

        sql = f"""
            INSERT INTO `{TABLE_RESULT}`
            (date, ticker, year_label, phase,
             sales_forecast, npm_forecast, asset_turnover, fin_leverage,
             roe_forecast, re, ri_spread, bv_start, ri, pv_ri,
             moat_label, rho, n_phase2, g_terminal,
             pv_all_ri, terminal_value, intrinsic_value,
             current_price, upside_pct, target_price,
             beta_raw, beta_blume, bv_source,
             revenue_quarters, forecast_model, forecast_date)
            VALUES
            (%(date)s, %(ticker)s, %(year_label)s, %(phase)s,
             %(sales_forecast)s, %(npm_forecast)s, %(asset_turnover)s, %(fin_leverage)s,
             %(roe_forecast)s, %(re)s, %(ri_spread)s, %(bv_start)s, %(ri)s, %(pv_ri)s,
             %(moat_label)s, %(rho)s, %(n_phase2)s, %(g_terminal)s,
             %(pv_all_ri)s, %(terminal_value)s, %(intrinsic_value)s,
             %(current_price)s, %(upside_pct)s, %(target_price)s,
             %(beta_raw)s, %(beta_blume)s, %(bv_source)s,
             %(revenue_quarters)s, %(forecast_model)s, %(forecast_date)s)
            ON DUPLICATE KEY UPDATE
                ri=VALUES(ri), roe_forecast=VALUES(roe_forecast),
                target_price=VALUES(target_price), upside_pct=VALUES(upside_pct),
                intrinsic_value=VALUES(intrinsic_value),
                moat_label=VALUES(moat_label)
        """
        conn = get_pymysql_conn(self.db_info)
        try:
            with conn.cursor() as cur:
                cur.executemany(sql, rows)
            conn.commit()
        except Exception:
            conn.rollback(); raise
        finally:
            conn.close()
        return len(rows)

    # ─────────────────────────────────────────────────────────
    # 9. 시각화 (한국 단위 조원)
    # ─────────────────────────────────────────────────────────
    def plot(self):
        if self.result_df is None or self.valuation is None:
            print("[WARN] run() 먼저 실행 필요"); return

        df = self.result_df.copy()
        v = self.valuation
        re = v["re"]
        UNIT, UNIT_LBL = 1e12, "조원"

        fig, axes = plt.subplots(2, 2, figsize=(16, 10))
        fig.suptitle(f"{self.ticker_dg}  Korea RIM Valuation  [{v['moat_label']}]",
                     fontsize=14, fontweight="bold")

        # (0,0) ROE vs Re
        ax = axes[0, 0]
        current_year = datetime.now().year
        wide = self._fs_wide
        roe_h_df = None
        if "net_income" in wide.columns and "total_equity" in wide.columns:
            df_h = wide[["net_income", "total_equity"]].dropna().sort_index()
            df_h = df_h[df_h["total_equity"] > 0].tail(20)
            if len(df_h) >= 4:
                roe_hist = []
                for i in range(3, len(df_h)):
                    ni_ttm = float(df_h["net_income"].iloc[i-3:i+1].sum())
                    eq_avg = float(df_h["total_equity"].iloc[i-3:i+1].mean())
                    if eq_avg <= 0: continue
                    dt = df_h.index[i]
                    yr = dt.year + (dt.month - 1) / 12
                    roe_hist.append({"yr": yr, "roe": ni_ttm / eq_avg})
                roe_h_df = pd.DataFrame(roe_hist)
                ax.plot(roe_h_df["yr"], roe_h_df["roe"] * 100,
                        color="#2980b9", lw=1.5, marker="o", ms=3,
                        label="Historical ROE (TTM)")
                ax.axvline(current_year, color="gray", lw=1, ls="--")

        ph1 = df[df["phase"].str.startswith("ph1")]
        ph2 = df[df["phase"] == "ph2"]
        ph1_x = [current_year + t for t in range(1, len(ph1) + 1)]
        ph2_x = [current_year + len(ph1) + t for t in range(1, len(ph2) + 1)]
        ax.plot(ph1_x, ph1["roe"] * 100, "s-", color="#27ae60", lw=2,
                label="Phase1 ROE (DuPont)")
        ax.plot(ph2_x, ph2["roe"] * 100, "^--", color="#8e44ad", lw=2,
                label="Phase2 ROE (AR(1))")
        ax.axhline(re * 100, color="#e74c3c", lw=1.5, ls=":",
                   label=f"Re={re*100:.1f}%")
        ax.axhline(0, color="black", lw=0.6)
        ax.set_title("ROE vs Cost of Equity (Re)")
        ax.set_ylabel("%")
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_:f"{x:.0f}%"))
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

        # (0,1) RI time-series
        ax = axes[0, 1]
        ph1_ri = ph1["ri"].values / UNIT
        ph2_ri = ph2["ri"].values / UNIT
        yr_ph1 = list(range(1, len(ph1)+1))
        yr_ph2 = list(range(len(ph1)+1, len(ph1)+len(ph2)+1))
        c_ph1 = ["#27ae60" if r >= 0 else "#e74c3c" for r in ph1_ri]
        c_ph2 = ["#8e44ad" if r >= 0 else "#c0392b" for r in ph2_ri]
        ax.bar(yr_ph1, ph1_ri, color=c_ph1, alpha=0.85, label="Phase1 RI", edgecolor="white")
        ax.bar(yr_ph2, ph2_ri, color=c_ph2, alpha=0.65, label="Phase2 RI", edgecolor="white")
        ax.axhline(0, color="black", lw=0.8)
        ax.axvline(len(ph1) + 0.5, color="gray", lw=1, ls=":")
        pv_tv_u = v["pv_tv"] / UNIT
        ax.axhline(pv_tv_u, color="#e67e22", lw=1.5, ls="--",
                   label=f"PV(TV)={pv_tv_u:.2f}{UNIT_LBL}")
        ax.set_title(f"Residual Income by Year ({UNIT_LBL})")
        ax.set_xlabel("Forecast Year"); ax.set_ylabel(UNIT_LBL)
        ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

        # (1,0) Value composition
        ax = axes[1, 0]
        bv0_u = v["bv0"] / UNIT
        pv_ri_u = v["pv_ri"] / UNIT
        pv_tv_u = v["pv_tv"] / UNIT
        iv_u = v["intrinsic_value"] / UNIT
        components = {"Book Value": bv0_u, "PV(RI)": pv_ri_u, "PV(TV)": pv_tv_u}
        colors_bar = {"Book Value": "#3498db", "PV(RI)": "#27ae60", "PV(TV)": "#e67e22"}
        bottom = 0.0
        for comp, val in components.items():
            if val > 0:
                ax.bar("Intrinsic Value", val, bottom=bottom,
                       color=colors_bar[comp], alpha=0.85, edgecolor="white",
                       label=f"{comp} {val:.2f}{UNIT_LBL}")
                ax.text(0, bottom + val/2, f"{comp}\n{val:.2f}{UNIT_LBL}",
                        ha="center", va="center", fontsize=9,
                        color="white", fontweight="bold")
                bottom += val
        if (not np.isnan(v["current_price"]) and not np.isnan(v["shares"])
            and v["shares"] > 0):
            mkt_cap_u = v["current_price"] * v["shares"] / UNIT
            ax.axhline(mkt_cap_u, color="#e74c3c", lw=2, ls="--",
                       label=f"Market Cap {mkt_cap_u:.2f}{UNIT_LBL}")
        ax.set_title(f"Intrinsic Value Composition (IV={iv_u:.2f}{UNIT_LBL})")
        ax.set_ylabel(UNIT_LBL); ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.3)

        # (1,1) Summary text
        ax = axes[1, 1]; ax.axis("off")
        tp = v["target_price"]; cp = v["current_price"]; up = v["upside_pct"]
        evs = v["eva_spread"]; rrs = v["roe_re_spread"]

        def _f(x, u="원", d=0):
            if x is None or (isinstance(x, float) and np.isnan(x)): return "N/A"
            if u == "원":  return f"{x:,.{d}f}원"
            if u == "조":  return f"{x/1e12:,.2f}조원"
            if u == "%":   return f"{x*100:.2f}%"
            return str(x)

        lines = [
            f"Current Price  : {_f(cp)}",
            f"Target Price   : {_f(tp)}",
            f"Upside         : {up:+.1f}%" if not np.isnan(up) else "Upside : N/A",
            "─" * 32,
            f"Book Value     : {_f(v['bv0'],'조')}  [{v['bv_source']}]",
            f"PV(RI)         : {_f(v['pv_ri'],'조')}",
            f"PV(TV)         : {_f(v['pv_tv'],'조')}",
            f"Intrinsic Val  : {_f(v['intrinsic_value'],'조')}",
            "─" * 32,
            f"Re             : {_f(re,'%')}",
            f"g_terminal     : {_f(v['g_terminal'],'%')}",
            f"β_blume        : {v['beta_blume']:.3f}" if not np.isnan(v['beta_blume']) else "β : N/A",
            f"Moat           : {v['moat_label']}",
            f"omega (AR1)    : {v['omega']:.3f}",
            f"Phase1 / Ph2   : {v['n_phase1']} / {v['n_phase2']} yr",
            "─" * 32,
            f"EVA spread     : {evs*100:+.1f}%" if not np.isnan(evs) else "EVA : N/A",
            f"ROE-Re spread  : {rrs*100:+.1f}%" if not np.isnan(rrs) else "ROE-Re : N/A",
        ]
        ax.text(0.05, 0.97, "\n".join(lines), transform=ax.transAxes,
                fontsize=9, verticalalignment="top", fontfamily="monospace",
                bbox=dict(boxstyle="round,pad=0.5", facecolor="#f8f9fa", alpha=0.8))
        plt.tight_layout(); plt.show()

    # ─────────────────────────────────────────────────────────
    # 10. 전체 실행
    # ─────────────────────────────────────────────────────────
    def run(self):
        self.load_sales()
        self.load_financials()
        self.load_re()
        self.compute_ri_path()
        self.compute_valuation()
        return self


# ── Wrapper ────────────────────────────────────────────────────
def process_one_ticker_rim_kr(ticker, engine, db_info, rf, e_rm, kospi_series,
                               verbose=False, run_date=None, save_db=True):
    run_date = run_date or datetime.now().strftime("%Y-%m-%d")
    try:
        m = KoreaRIMModel(ticker=ticker, engine=engine, db_info=db_info,
                          rf=rf, e_rm=e_rm, kospi_series=kospi_series,
                          verbose=verbose)
        m.run()
        rows = m.save_to_db(run_date) if save_db else 0
        v = m.valuation
        return {
            "status": "ok", "ticker": m.ticker_dg,
            "target_price": v.get("target_price", np.nan),
            "current_price": v.get("current_price", np.nan),
            "upside_pct":   v.get("upside_pct", np.nan),
            "re":           v.get("re", np.nan),
            "moat_label":   v.get("moat_label", ""),
            "omega":        v.get("omega", np.nan),
            "n_phase2":     v.get("n_phase2", 0),
            "eva_spread":   v.get("eva_spread", np.nan),
            "bv_source":    v.get("bv_source", ""),
            "rows_saved":   rows,
            "report":       m.report,
        }
    except Exception as e:
        return {
            "status": "fail", "ticker": to_dg_ticker(ticker),
            "target_price": np.nan, "current_price": np.nan, "upside_pct": np.nan,
            "re": np.nan, "moat_label": "", "omega": np.nan, "n_phase2": 0,
            "eva_spread": np.nan, "bv_source": "", "rows_saved": 0,
            "report": None, "msg": str(e)[:150],
        }
    finally:
        gc.collect()


print("[OK] KoreaRIMModel 정의 완료")

# ── 단일 티커 테스트 (삼성전자) ─────────────────────────────────
TEST_TICKER = "A092870"
print(f"\n[테스트] {TEST_TICKER}")
_res = process_one_ticker_rim_kr(TEST_TICKER, engine, db_info, RF, E_RM, KOSPI_PX,
                                  verbose=True, save_db=False)
print(f"\n  status={_res['status']}  moat={_res.get('moat_label','')}  "
      f"omega={_res.get('omega',np.nan):.3f}")
if _res["status"] == "ok":
    tp_s = f"{_res['target_price']:,.0f}원" if not np.isnan(_res['target_price']) else "N/A"
    cp_s = f"{_res['current_price']:,.0f}원" if not np.isnan(_res['current_price']) else "N/A"
    up_s = f"{_res['upside_pct']:+.1f}%" if not np.isnan(_res['upside_pct']) else "N/A"
    print(f"  TP={tp_s}  CP={cp_s}  Upside={up_s}  BV src={_res['bv_source']}")
    print()
    print("─" * 60); print("데이터 품질 리포트:"); print("─" * 60)
    print(_res["report"].summary())
    display(_res["report"].to_dataframe())
    # 시각화
    _m = KoreaRIMModel(ticker=TEST_TICKER, engine=engine, db_info=db_info,
                       rf=RF, e_rm=E_RM, kospi_series=KOSPI_PX, verbose=False).run()
    _m.plot()
else:
    print(f"  msg={_res.get('msg','')}")


[OK] KoreaRIMModel 정의 완료

[테스트] A124500
[14:13:35][A124500] Sales actual=46Q forecast=8Q (Ensemble)
[14:13:35][A124500] FS wide shape=(67, 35)
[14:13:38][A124500] Re=10.5643%  β_raw=0.946 β_blume=0.964  Rf=3.817% ERP=7.000%
[14:13:38][A124500] DuPont  NPM=0.0223(ols)  AT=3.621(TTM)  FL=2.68
[14:13:43][A124500] EVA: ROIC=61.57% WACC=8.88% spread=+52.69% n_pos=12/20  ROE-Re=+39.1%
[14:13:43][A124500] Retention=0.577  payout=0.423
[14:13:43][A124500] Retention=0.577  payout=0.423
[14:13:43][A124500] RI path: Ph1=6yr Ph2=28yr  RI_last=0.24조  [Wide moat ω=0.98 PVf=7.53]
[14:13:45][A124500] RIM  BV=0.51조 PV(RI)=2.80조 PV(TV)=0.86조 IV=4.17조  TP=179,706원 CP=53,200원 Up=+237.8%  [Wide moat]

  status=ok  moat=Wide moat  omega=0.980
  TP=179,706원  CP=53,200원  Upside=+237.8%  BV src=Equity

────────────────────────────────────────────────────────────
데이터 품질 리포트:
────────────────────────────────────────────────────────────
[A124500]  ok=10  fallback_median=0  fallback_zero=0  missing=0  warning=0  m

,field,status,n_obs,value,r2,note
0,revenue_actual,ok,46,NaN,NaN,
1,revenue_forecast,ok,8,NaN,NaN,model=Ensemble
2,fs_core,ok,46,NaN,NaN,
3,beta,ok,2450,9.638939e-01,0.092875,β_raw=0.946
4,npm,ok,46,2.233696e-02,0.750672,OLS slope
5,asset_turnover,ok,43,3.621378e+00,NaN,
6,financial_leverage,ok,53,2.681119e+00,NaN,
7,bv0,ok,0,5.118079e+11,NaN,
8,retention,ok,2,5.765932e-01,NaN,payout_med=42.34%
9,retention,ok,2,5.765932e-01,NaN,payout_med=42.34%


## Cell 5.5 · RIM 진단 (단일 종목 상세 분석)

특정 종목의 RIM 분해 내역 (Re 구성요소, Phase 1 DuPont, Phase 2 AR(1) decay, Value Breakdown) 상세 확인용. 티커만 변경해서 재실행.


In [ ]:
DIAG_TICKER = "A005930"    # ← 변경해서 분석할 종목 지정

print(f"[Diagnostics] Running KoreaRIMModel for {DIAG_TICKER} ...")
_rm = KoreaRIMModel(ticker=DIAG_TICKER, engine=engine, db_info=db_info,
                    rf=RF, e_rm=E_RM, kospi_series=KOSPI_PX, verbose=True)
_rm.run()

v = _rm.valuation
df = _rm.result_df.copy()

SEP = "=" * 70
print(f"\n{SEP}")
print(f"  {DIAG_TICKER}  Korea RIM Diagnostics")
print(SEP)

# [1] Cost of Equity 분해
print(f"\n[1] Cost of Equity")
print(f"    Rf              : {RF*100:.3f}%  (BOK 10Y 국고채)")
print(f"    ERP             : {ERP*100:.3f}%  (method={ERP_METHOD})")
print(f"    β_raw           : {v['beta_raw']:.3f}" if not np.isnan(v['beta_raw']) else "    β_raw : fallback 1.0")
print(f"    β_blume         : {v['beta_blume']:.3f}  (= 0.67×|β| + 0.33)")
print(f"    Re = Rf + β_blume × ERP  =  {v['re']*100:.3f}%")
print(f"    clip 범위       : [{RE_FLOOR:.0%}, {RE_CAP:.0%}]")
print(f"    g_terminal      : {v['g_terminal']*100:.2f}%")
print(f"    Re - g          : {(v['re']-v['g_terminal'])*100:.3f}%")

# [2] Phase 1 DuPont
ph1_all = df[df["phase"].str.startswith("ph1")]
ph1_fc  = df[df["phase"] == "ph1"]
ph1_ext = df[df["phase"] == "ph1e"]
print(f"\n[2] Phase 1 — DuPont ROE  "
      f"[실제 예측 {len(ph1_fc)}yr + plateau 연장 {len(ph1_ext)}yr = 총 {len(ph1_all)}yr]")
for _, row in ph1_all.iterrows():
    tag = "" if row["phase"] == "ph1" else " [plateau]"
    bv_u = row["bv_start"] / 1e12
    ri_u = row["ri"] / 1e12
    print(f"    Year {int(row['year'])}{tag}  "
          f"Sales={row['sales_annual']/1e12:.2f}조  "
          f"NPM={row['npm']*100:.1f}%  AT={row['at']:.2f}  FL={row['fl']:.1f}  "
          f"ROE={row['roe']*100:.1f}%  BV={bv_u:.2f}조  RI={ri_u:+.3f}조")

# [3] Phase 2 AR(1)
print(f"\n[3] Phase 2 — AR(1) RI Decay  (ω={v['omega']:.3f}, {v['n_phase2']}yr)")
ph2 = df[df["phase"] == "ph2"]
for _, row in ph2.iterrows():
    ri_u = row["ri"] / 1e12
    bv_u = row["bv_start"] / 1e12
    print(f"    Year {int(row['year'])}  "
          f"RI={ri_u:+.3f}조  ROE_impl={row['roe']*100:.1f}%  "
          f"spread={row['ri_spread']*100:+.1f}%  BV={bv_u:.2f}조")

# [4] TV
print(f"\n[4] Terminal Value")
ri_last_u = _rm._ri_last / 1e12
tv_u = v["terminal_value"] / 1e12
pv_tv_u = v["pv_tv"] / 1e12
print(f"    RI_last          : {ri_last_u:+.3f}조")
print(f"    TV = RI_last×(1+g)/(Re-g) = {tv_u:.3f}조" if v["terminal_value"] > 0 else "    TV = 0 (RI_last ≤ 0)")
print(f"    PV(TV)           : {pv_tv_u:.3f}조")

# [5] Value Breakdown
print(f"\n[5] Value Breakdown")
iv_u = v["intrinsic_value"] / 1e12
bv0_u = v["bv0"] / 1e12
pv_ri_u = v["pv_ri"] / 1e12
bv_wt = bv0_u / iv_u * 100 if iv_u else np.nan
ri_wt = pv_ri_u / iv_u * 100 if iv_u else np.nan
tv_wt = pv_tv_u / iv_u * 100 if iv_u else np.nan
print(f"    Book Value      : {bv0_u:>7.3f}조 ({bv_wt:>5.1f}%)  [{v['bv_source']}]")
print(f"    PV(RI)          : {pv_ri_u:>7.3f}조 ({ri_wt:>5.1f}%)")
print(f"    PV(TV)          : {pv_tv_u:>7.3f}조 ({tv_wt:>5.1f}%)")
print(f"    ─────────────────────────────────")
print(f"    Intrinsic Value : {iv_u:>7.3f}조")

# [6] Target Price
print(f"\n[6] Target Price")
tp_s = f"{v['target_price']:,.0f}원" if not np.isnan(v["target_price"]) else "N/A"
cp_s = f"{v['current_price']:,.0f}원" if not np.isnan(v["current_price"]) else "N/A"
up_s = f"{v['upside_pct']:+.1f}%" if not np.isnan(v["upside_pct"]) else "N/A"
sh_s = f"{v['shares']:,.0f}주" if not np.isnan(v["shares"]) else "N/A"
print(f"    Shares          : {sh_s}")
print(f"    TP = IV / shares = {tp_s}")
print(f"    Current Price   : {cp_s}")
print(f"    Upside          : {up_s}")

# [7] EVA / Moat
print(f"\n[7] EVA / Moat 분류 (6-Tier)")
evs = v["eva_spread"]
rrs = v["roe_re_spread"]
print(f"    EVA spread      : {evs*100:+.2f}%" if not np.isnan(evs) else "    EVA : N/A")
print(f"    ROE-Re spread   : {rrs*100:+.2f}%" if not np.isnan(rrs) else "    ROE-Re : N/A")
print(f"    Moat grade      : {v['moat_label']}")
print(f"    omega (AR1)     : {v['omega']:.3f}")
print(f"    Phase1 period   : {v['n_phase1']} yr  (기본 2yr + {PHASE1_EXTRA_YR.get(v['moat_label'],1)}yr plateau)")
print(f"    Phase2 period   : {v['n_phase2']} yr")

# [8] 데이터 품질
print(f"\n[8] 데이터 품질")
print(_rm.report.summary())
display(_rm.report.to_dataframe())

# 시각화
print(f"\n[시각화]")
_rm.plot()


## Cell 6 · 배치 실행

In [ ]:
RUN_TICKERS = KOREA_TICKER_LIST[TICKER_START:TICKER_END]
total = len(RUN_TICKERS)
run_date = datetime.now().strftime("%Y-%m-%d")

done_set = set()
if SKIP_DONE and os.path.exists(DONE_PATH):
    with open(DONE_PATH, encoding="utf-8") as f:
        done_set = {l.strip() for l in f if l.strip()}

ok_cnt = skip_cnt = fail_cnt = 0
results, all_reports = [], []
t0 = time.time()

log("BATCH", f"RIM 배치 시작: {total:,}개  run_date={run_date}")
print("=" * 80)

for idx, ticker in enumerate(RUN_TICKERS, 1):
    pct = idx / total * 100
    prefix = f"[{idx:>5}/{total}] ({pct:5.1f}%) {ticker:<8}"

    if SKIP_DONE and ticker in done_set:
        print(f"{prefix} SKIP", flush=True); skip_cnt += 1; continue

    res = process_one_ticker_rim_kr(
        ticker, engine, db_info, RF, E_RM, KOSPI_PX,
        verbose=False, run_date=run_date, save_db=True)
    results.append(res)
    if res["report"] is not None:
        all_reports.append(res["report"])

    if res["status"] == "ok":
        tp = res["target_price"]; up = res["upside_pct"]
        tp_s = f"TP={tp:,.0f}원" if not np.isnan(tp) else "TP=N/A"
        up_s = f"↑{up:+.1f}%" if not np.isnan(up) else ""
        print(f"{prefix} OK  {tp_s} {up_s}  Re={res['re']:.3%}  [{res['moat_label']}] ω={res['omega']:.2f}", flush=True)
        with open(DONE_PATH, "a", encoding="utf-8") as f:
            f.write(ticker + "\n")
        ok_cnt += 1
    else:
        print(f"{prefix} FAIL  {res.get('msg','')}", flush=True)
        with open(FAIL_PATH, "a", encoding="utf-8") as f:
            f.write(f"{ticker}\t{res.get('msg','')}\n")
        fail_cnt += 1

elapsed = time.time() - t0
print("=" * 80)
log("BATCH", f"완료  OK={ok_cnt}  SKIP={skip_cnt}  FAIL={fail_cnt}  "
             f"경과={elapsed:.0f}s  평균={elapsed/max(ok_cnt+fail_cnt,1):.1f}s/ticker")

# 품질 로그 저장
if all_reports:
    n_q = save_quality_report_to_db(
        all_reports, db_info, table_name=TABLE_QUALITY,
        run_date=run_date, model_name="RIM")
    log("QUALITY", f"품질 로그 {n_q:,}건 → {TABLE_QUALITY}")

# 요약
if results:
    summary = pd.DataFrame([{
        "ticker": r["ticker"], "re": r["re"],
        "moat": r["moat_label"], "omega": r["omega"],
        "tp": r["target_price"], "cp": r["current_price"],
        "upside": r["upside_pct"], "bv_src": r["bv_source"],
        "status": r["status"],
    } for r in results])
    ok_summary = summary[summary["status"] == "ok"].sort_values("upside", ascending=False)
    print("\n[상위 업사이드 TOP 20]")
    display(ok_summary.head(20))


## Cell 7 · 결과 조회 & 시각화

In [ ]:
# ── 7-1. DB 현황 ──────────────────────────────────────
conn = get_pymysql_conn(db_info)
try:
    with conn.cursor() as cur:
        cur.execute(f"""
            SELECT DATE(date) AS run_date,
                   COUNT(DISTINCT ticker) AS tickers,
                   COUNT(*) AS total_rows,
                   AVG(target_price) AS avg_tp,
                   AVG(upside_pct) AS avg_upside,
                   AVG(re) AS avg_re
            FROM {TABLE_RESULT}
            GROUP BY DATE(date)
            ORDER BY run_date DESC LIMIT 10
        """)
        summary_df = pd.DataFrame(cur.fetchall())
finally:
    conn.close()

print("=" * 70); print(f"[DB 현황] {TABLE_RESULT}"); print("=" * 70)
display(summary_df)

# ── 7-2. 최신 평가일 결과 ─────────────────────────────
conn = get_pymysql_conn(db_info)
try:
    with conn.cursor() as cur:
        cur.execute(f"SELECT MAX(date) AS m FROM {TABLE_RESULT}")
        max_date = cur.fetchone()["m"]
        if max_date:
            cur.execute(f"""
                SELECT ticker,
                       MAX(target_price) AS target_price,
                       MAX(current_price) AS current_price,
                       MAX(upside_pct) AS upside_pct,
                       MAX(re) AS re,
                       MAX(rho) AS omega,
                       MAX(n_phase2) AS n_phase2,
                       MAX(moat_label) AS moat,
                       MAX(bv_source) AS bv_source,
                       MAX(intrinsic_value) AS intrinsic_value
                FROM {TABLE_RESULT}
                WHERE date = %s
                  AND target_price IS NOT NULL
                  AND current_price IS NOT NULL
                GROUP BY ticker
                ORDER BY upside_pct DESC
            """, (max_date,))
            results_df = pd.DataFrame(cur.fetchall())
        else:
            results_df = pd.DataFrame()
finally:
    conn.close()

if not results_df.empty:
    print(f"\n[최신 평가일: {max_date}] 총 {len(results_df):,}개")
    print("\n🔼 업사이드 TOP 20"); display(results_df.head(20))
    print("\n🔽 다운사이드 TOP 20"); display(results_df.tail(20))

    # 시각화
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle(f"Korea RIM Valuation Summary ({max_date})", fontsize=13)

    ax = axes[0]
    upside_clean = results_df["upside_pct"].dropna().clip(-200, 200)
    ax.hist(upside_clean, bins=40, color="#3498db", edgecolor="white", alpha=0.8)
    ax.axvline(0, color="black", lw=1)
    ax.axvline(20, color="green", lw=1, ls="--", label="+20%")
    ax.set_title("Upside Distribution")
    ax.set_xlabel("Upside (%)"); ax.legend(); ax.grid(alpha=0.3)

    ax = axes[1]
    grade = pd.cut(results_df["upside_pct"],
                   bins=[-np.inf, -20, 0, 20, 50, np.inf],
                   labels=["Strong Sell", "Sell", "Hold", "Buy", "Strong Buy"])
    grade.value_counts().sort_index().plot(
        kind="barh", ax=ax,
        color=["#c0392b", "#e74c3c", "#f39c12", "#2ecc71", "#27ae60"])
    ax.set_title("Valuation Grade"); ax.grid(axis="x", alpha=0.3)

    ax = axes[2]
    moat_cnts = results_df["moat"].value_counts()
    colors_moat = {
        "Exceptional moat": "#27ae60", "Wide moat": "#2ecc71",
        "Wide-Narrow moat": "#f1c40f", "Narrow moat": "#f39c12",
        "Some moat": "#e67e22", "No moat": "#c0392b",
        "Unknown (fallback)": "#95a5a6",
    }
    bar_colors = [colors_moat.get(m, "#95a5a6") for m in moat_cnts.index]
    ax.barh(range(len(moat_cnts)), moat_cnts.values, color=bar_colors, alpha=0.85)
    ax.set_yticks(range(len(moat_cnts)))
    ax.set_yticklabels(moat_cnts.index, fontsize=9)
    ax.set_title("Moat Distribution"); ax.grid(axis="x", alpha=0.3)

    plt.tight_layout(); plt.show()
else:
    print("[INFO] 데이터 없음 — Cell 6 배치 실행 필요")

# ── 7-3. 종목별 RIM 평가 이력 ─────────────────────────
def get_rim_history(ticker, db_info, table_name=TABLE_RESULT):
    tk = to_dg_ticker(ticker)
    sql = f"""
        SELECT date,
               MAX(target_price) AS target_price,
               MAX(current_price) AS current_price,
               MAX(upside_pct) AS upside_pct,
               MAX(re) AS re,
               MAX(moat_label) AS moat,
               MAX(intrinsic_value) AS iv
        FROM `{table_name}`
        WHERE ticker = %s
        GROUP BY date
        ORDER BY date DESC
    """
    conn = get_pymysql_conn(db_info)
    try:
        with conn.cursor() as cur:
            cur.execute(sql, (tk,))
            return pd.DataFrame(cur.fetchall())
    finally:
        conn.close()

print("\n[예시] 삼성전자 RIM 평가 이력")
display(get_rim_history("A005930", db_info))


## Cell 8 · 데이터 품질 진단 (호영님 결정 #5)

배치 실행 시 모든 fallback 내역이 `korea_valuation_quality_log` 에 저장됩니다. 이 셀에서 어떤 종목의 어떤 항목이 어떤 이유로 fallback 되었는지 확인할 수 있습니다.


In [ ]:
conn = get_pymysql_conn(db_info)
try:
    with conn.cursor() as cur:
        # [1] 항목별 status 분포
        cur.execute(f"""
            SELECT field, status, COUNT(*) AS cnt,
                   AVG(r_squared) AS avg_r2, AVG(value) AS avg_value
            FROM {TABLE_QUALITY}
            WHERE model='RIM'
              AND date=(SELECT MAX(date) FROM {TABLE_QUALITY} WHERE model='RIM')
            GROUP BY field, status
            ORDER BY field, status
        """)
        df_status = pd.DataFrame(cur.fetchall())

        # [2] 항목별 fallback 비율
        cur.execute(f"""
            SELECT field,
                   SUM(status='ok') AS ok_cnt,
                   SUM(status='fallback_median') AS fb_med,
                   SUM(status='fallback_zero') AS fb_zero,
                   SUM(status='missing') AS miss,
                   COUNT(*) AS total,
                   ROUND(SUM(status LIKE 'fallback%')/COUNT(*)*100, 1) AS fb_pct
            FROM {TABLE_QUALITY}
            WHERE model='RIM'
              AND date=(SELECT MAX(date) FROM {TABLE_QUALITY} WHERE model='RIM')
            GROUP BY field
            ORDER BY fb_pct DESC
        """)
        df_field = pd.DataFrame(cur.fetchall())

        # [3] 문제 종목 TOP 30
        cur.execute(f"""
            SELECT ticker,
                   SUM(status LIKE 'fallback%') AS fb_cnt,
                   SUM(status='missing') AS miss_cnt,
                   GROUP_CONCAT(DISTINCT field ORDER BY field SEPARATOR ',') AS issues
            FROM {TABLE_QUALITY}
            WHERE model='RIM'
              AND date=(SELECT MAX(date) FROM {TABLE_QUALITY} WHERE model='RIM')
              AND (status LIKE 'fallback%' OR status='missing')
            GROUP BY ticker
            HAVING fb_cnt + miss_cnt >= 3
            ORDER BY (fb_cnt + miss_cnt*2) DESC
            LIMIT 30
        """)
        df_problem = pd.DataFrame(cur.fetchall())

        # [4] 음수 BV (IC fallback) 종목 리스트
        cur.execute(f"""
            SELECT DISTINCT ticker, MAX(intrinsic_value) AS iv,
                   MAX(target_price) AS tp, MAX(upside_pct) AS up
            FROM {TABLE_RESULT}
            WHERE bv_source = 'IC_fallback'
              AND date=(SELECT MAX(date) FROM {TABLE_RESULT})
            GROUP BY ticker
            ORDER BY upside_pct DESC
        """)
        df_ic = pd.DataFrame(cur.fetchall())
finally:
    conn.close()

print("=" * 70); print("[1] 항목별 status 분포 (최신 RIM 배치)"); print("=" * 70)
display(df_status)

print("\n" + "=" * 70); print("[2] 항목별 fallback 비율"); print("=" * 70)
display(df_field)

print("\n" + "=" * 70); print("[3] 데이터 문제가 많은 종목 TOP 30"); print("=" * 70)
display(df_problem)

print("\n" + "=" * 70); print("[4] 음수 BV → IC fallback 종목 (신뢰도 주의)"); print("=" * 70)
display(df_ic)


def inspect_rim_quality(ticker, run_date=None):
    """특정 종목의 RIM 데이터 품질 상세."""
    tk = to_dg_ticker(ticker)
    conn = get_pymysql_conn(db_info)
    try:
        with conn.cursor() as cur:
            if run_date:
                cur.execute(f"""SELECT date, field, status, n_obs, value, r_squared, note
                                  FROM {TABLE_QUALITY}
                                  WHERE ticker=%s AND date=%s AND model='RIM'
                                  ORDER BY field""", (tk, run_date))
            else:
                cur.execute(f"""SELECT date, field, status, n_obs, value, r_squared, note
                                  FROM {TABLE_QUALITY}
                                  WHERE ticker=%s AND model='RIM'
                                    AND date=(SELECT MAX(date) FROM {TABLE_QUALITY}
                                               WHERE ticker=%s AND model='RIM')
                                  ORDER BY field""", (tk, tk))
            return pd.DataFrame(cur.fetchall())
    finally:
        conn.close()

print("\n[예시] 삼성전자 RIM 품질 상세")
display(inspect_rim_quality("A005930"))
